# 13 - Ontology Enrichment

Loads cached external ontologies and extracts semantic metadata for use in RDF→LPG translation.

**Prerequisites:** Attach Fabric Environment `env_rdf_jena` with `jena-shaded-4.10.0.jar`
(Same environment used by `01_rdf_parser_jena.ipynb`)

**Input:** Cached ontology files from `Files/cache/external_ontologies/`
**Output:** `Files/cache/ontology_metadata.json` with:
- Labels (`rdfs:label`, `skos:prefLabel`) for classes and properties
- Class hierarchy (`rdfs:subClassOf`)
- Property domains/ranges (`rdfs:domain`, `rdfs:range`)

**Usage:** Run after `12_external_ontology_fetcher.ipynb` to enrich translation metadata.

In [ ]:
# Configuration
LAKEHOUSE_PATH = "/lakehouse/default/Files"
CACHE_DIR = f"{LAKEHOUSE_PATH}/cache/external_ontologies"
OUTPUT_PATH = f"{LAKEHOUSE_PATH}/cache/ontology_metadata.json"

# Supported ontology file extensions
SUPPORTED_EXTENSIONS = [".ttl", ".rdf", ".nt", ".n3", ".jsonld"]

print(f"Cache directory: {CACHE_DIR}")
print(f"Output path: {OUTPUT_PATH}")

In [ ]:
import os
import json
import glob
from typing import Dict, List, Set, Optional, Tuple
from collections import defaultdict

print("Imports ready.")

In [ ]:
# Initialize Jena via Py4J (Spark's gateway)
# Requires: Fabric Environment with jena-shaded-4.10.0.jar attached

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext
gateway = sc._gateway

# Import Jena classes via Py4J
try:
    ModelFactory = gateway.jvm.org.apache.jena.rdf.model.ModelFactory
    RDFDataMgr = gateway.jvm.org.apache.jena.riot.RDFDataMgr
    RDFS = gateway.jvm.org.apache.jena.vocabulary.RDFS
    RDF = gateway.jvm.org.apache.jena.vocabulary.RDF
    OWL = gateway.jvm.org.apache.jena.vocabulary.OWL
    
    # SKOS namespace (not in Jena core)
    SKOS_PREF_LABEL = "http://www.w3.org/2004/02/skos/core#prefLabel"
    SKOS_ALT_LABEL = "http://www.w3.org/2004/02/skos/core#altLabel"
    
    print("✓ Jena initialized via Py4J.")
except Exception as e:
    print(f"✗ Failed to initialize Jena: {e}")
    print("  Make sure the Environment 'env_rdf_jena' is attached to this notebook.")

In [ ]:
# Discover cached ontology files
def discover_ontology_files(cache_dir: str) -> List[str]:
    """Find all ontology files in the cache directory."""
    files = []
    if not os.path.exists(cache_dir):
        print(f"Cache directory does not exist: {cache_dir}")
        return files
    
    for ext in SUPPORTED_EXTENSIONS:
        pattern = f"{cache_dir}/*{ext}"
        matches = glob.glob(pattern)
        files.extend(matches)
    
    return sorted(files)

ontology_files = discover_ontology_files(CACHE_DIR)
print(f"Found {len(ontology_files)} ontology files:")
for f in ontology_files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  - {os.path.basename(f)} ({size_kb:.1f} KB)")

In [ ]:
def load_ontology(filepath: str):
    """Load an ontology file into a Jena Model."""
    model = ModelFactory.createDefaultModel()
    try:
        # Jena auto-detects format from file extension
        RDFDataMgr.read(model, filepath)
        return model
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

def get_uri_str(resource) -> Optional[str]:
    """Safely get URI string from a Jena resource."""
    try:
        if resource is None:
            return None
        if resource.isURIResource():
            return str(resource.getURI())
        return None
    except:
        return None

def get_literal_str(node) -> Optional[str]:
    """Safely get string value from a Jena literal."""
    try:
        if node is None:
            return None
        if node.isLiteral():
            return str(node.asLiteral().getString())
        return None
    except:
        return None

print("Helper functions defined.")

In [ ]:
def extract_labels(model) -> Dict[str, Dict[str, str]]:
    """
    Extract labels for all resources in the model.
    Returns: {uri: {lang: label, ...}, ...}
    
    Checks: rdfs:label, skos:prefLabel, skos:altLabel
    """
    labels = defaultdict(dict)
    
    # rdfs:label
    rdfs_label_prop = model.getProperty(str(RDFS.label.getURI()))
    stmt_iter = model.listStatements(None, rdfs_label_prop, None)
    while stmt_iter.hasNext():
        stmt = stmt_iter.next()
        uri = get_uri_str(stmt.getSubject())
        if uri:
            obj = stmt.getObject()
            if obj.isLiteral():
                lit = obj.asLiteral()
                lang = str(lit.getLanguage()) if lit.getLanguage() else "default"
                labels[uri][lang] = str(lit.getString())
    
    # skos:prefLabel
    skos_pref_prop = model.getProperty(SKOS_PREF_LABEL)
    stmt_iter = model.listStatements(None, skos_pref_prop, None)
    while stmt_iter.hasNext():
        stmt = stmt_iter.next()
        uri = get_uri_str(stmt.getSubject())
        if uri:
            obj = stmt.getObject()
            if obj.isLiteral():
                lit = obj.asLiteral()
                lang = str(lit.getLanguage()) if lit.getLanguage() else "default"
                # Prefer skos:prefLabel over rdfs:label if not already set
                if lang not in labels[uri]:
                    labels[uri][lang] = str(lit.getString())
    
    return dict(labels)

print("Label extraction function defined.")

In [ ]:
def extract_class_hierarchy(model) -> Dict[str, List[str]]:
    """
    Extract class hierarchy (rdfs:subClassOf relationships).
    Returns: {class_uri: [parent_uri, ...], ...}
    """
    hierarchy = defaultdict(list)
    
    subclass_prop = model.getProperty(str(RDFS.subClassOf.getURI()))
    stmt_iter = model.listStatements(None, subclass_prop, None)
    
    while stmt_iter.hasNext():
        stmt = stmt_iter.next()
        child_uri = get_uri_str(stmt.getSubject())
        parent_uri = get_uri_str(stmt.getObject())
        
        if child_uri and parent_uri:
            # Skip owl:Thing and similar top-level classes
            if "owl#Thing" not in parent_uri and "rdfs#Resource" not in parent_uri:
                hierarchy[child_uri].append(parent_uri)
    
    return dict(hierarchy)

print("Hierarchy extraction function defined.")

In [ ]:
def extract_property_metadata(model) -> Dict[str, Dict]:
    """
    Extract property domains and ranges.
    Returns: {property_uri: {"domains": [...], "ranges": [...]}, ...}
    """
    properties = defaultdict(lambda: {"domains": [], "ranges": []})
    
    # rdfs:domain
    domain_prop = model.getProperty(str(RDFS.domain.getURI()))
    stmt_iter = model.listStatements(None, domain_prop, None)
    while stmt_iter.hasNext():
        stmt = stmt_iter.next()
        prop_uri = get_uri_str(stmt.getSubject())
        domain_uri = get_uri_str(stmt.getObject())
        if prop_uri and domain_uri:
            properties[prop_uri]["domains"].append(domain_uri)
    
    # rdfs:range
    range_prop = model.getProperty(str(RDFS.range.getURI()))
    stmt_iter = model.listStatements(None, range_prop, None)
    while stmt_iter.hasNext():
        stmt = stmt_iter.next()
        prop_uri = get_uri_str(stmt.getSubject())
        range_uri = get_uri_str(stmt.getObject())
        if prop_uri and range_uri:
            properties[prop_uri]["ranges"].append(range_uri)
    
    # Filter out properties with no domain or range info
    return {k: v for k, v in properties.items() if v["domains"] or v["ranges"]}

print("Property metadata extraction function defined.")

In [ ]:
def extract_classes(model) -> Set[str]:
    """
    Extract all class URIs from the model.
    Looks for: rdf:type owl:Class, rdfs:Class
    """
    classes = set()
    
    rdf_type = model.getProperty(str(RDF.type.getURI()))
    
    # owl:Class instances
    owl_class = model.getResource(str(OWL.Class.getURI()))
    stmt_iter = model.listStatements(None, rdf_type, owl_class)
    while stmt_iter.hasNext():
        stmt = stmt_iter.next()
        uri = get_uri_str(stmt.getSubject())
        if uri:
            classes.add(uri)
    
    # rdfs:Class instances
    rdfs_class = model.getResource(str(RDFS.Class.getURI()))
    stmt_iter = model.listStatements(None, rdf_type, rdfs_class)
    while stmt_iter.hasNext():
        stmt = stmt_iter.next()
        uri = get_uri_str(stmt.getSubject())
        if uri:
            classes.add(uri)
    
    return classes

def extract_properties(model) -> Set[str]:
    """
    Extract all property URIs from the model.
    Looks for: rdf:type owl:ObjectProperty, owl:DatatypeProperty, rdf:Property
    """
    properties = set()
    
    rdf_type = model.getProperty(str(RDF.type.getURI()))
    
    # owl:ObjectProperty
    obj_prop = model.getResource(str(OWL.ObjectProperty.getURI()))
    stmt_iter = model.listStatements(None, rdf_type, obj_prop)
    while stmt_iter.hasNext():
        stmt = stmt_iter.next()
        uri = get_uri_str(stmt.getSubject())
        if uri:
            properties.add(uri)
    
    # owl:DatatypeProperty
    data_prop = model.getResource(str(OWL.DatatypeProperty.getURI()))
    stmt_iter = model.listStatements(None, rdf_type, data_prop)
    while stmt_iter.hasNext():
        stmt = stmt_iter.next()
        uri = get_uri_str(stmt.getSubject())
        if uri:
            properties.add(uri)
    
    # rdf:Property
    rdf_property = model.getResource(str(RDF.Property.getURI()))
    stmt_iter = model.listStatements(None, rdf_type, rdf_property)
    while stmt_iter.hasNext():
        stmt = stmt_iter.next()
        uri = get_uri_str(stmt.getSubject())
        if uri:
            properties.add(uri)
    
    return properties

print("Class and property extraction functions defined.")

In [ ]:
# Process all ontology files
all_labels = {}
all_hierarchy = {}
all_property_metadata = {}
all_classes = set()
all_properties = set()
source_files = []

print(f"Processing {len(ontology_files)} ontology files...\n")

for filepath in ontology_files:
    filename = os.path.basename(filepath)
    print(f"Processing: {filename}")
    
    model = load_ontology(filepath)
    if model is None:
        print(f"  ✗ Failed to load\n")
        continue
    
    triple_count = model.size()
    print(f"  Triples: {triple_count}")
    
    # Extract metadata
    labels = extract_labels(model)
    hierarchy = extract_class_hierarchy(model)
    prop_meta = extract_property_metadata(model)
    classes = extract_classes(model)
    properties = extract_properties(model)
    
    print(f"  Labels: {len(labels)}")
    print(f"  Classes: {len(classes)}")
    print(f"  Properties: {len(properties)}")
    print(f"  Hierarchy entries: {len(hierarchy)}")
    print(f"  Property domains/ranges: {len(prop_meta)}")
    
    # Merge into global collections
    all_labels.update(labels)
    all_hierarchy.update(hierarchy)
    all_property_metadata.update(prop_meta)
    all_classes.update(classes)
    all_properties.update(properties)
    
    source_files.append({
        "filename": filename,
        "triples": triple_count,
        "labels": len(labels),
        "classes": len(classes),
        "properties": len(properties)
    })
    
    # Close model to free memory
    model.close()
    print()

print("=" * 50)
print(f"Total labels: {len(all_labels)}")
print(f"Total classes: {len(all_classes)}")
print(f"Total properties: {len(all_properties)}")
print(f"Total hierarchy entries: {len(all_hierarchy)}")
print(f"Total property metadata: {len(all_property_metadata)}")

In [ ]:
def get_best_label(labels_dict: Dict[str, str], preferred_langs: List[str] = ["en", "default", "nl"]) -> Optional[str]:
    """
    Get the best label from a language-keyed dict.
    Tries preferred languages in order, falls back to first available.
    """
    if not labels_dict:
        return None
    
    for lang in preferred_langs:
        if lang in labels_dict:
            return labels_dict[lang]
    
    # Fall back to first available
    return next(iter(labels_dict.values()))

# Build simplified label lookup (uri -> best_label)
simple_labels = {}
for uri, langs in all_labels.items():
    label = get_best_label(langs)
    if label:
        simple_labels[uri] = label

print(f"Built simplified labels for {len(simple_labels)} URIs")

# Sample labels
print("\nSample labels:")
for uri, label in list(simple_labels.items())[:10]:
    short_uri = uri.split("/")[-1].split("#")[-1][:30]
    print(f"  {short_uri}: {label[:50]}")

In [ ]:
from datetime import datetime

# Build output structure
output = {
    "generated_at": datetime.utcnow().isoformat(),
    "source_files": source_files,
    "statistics": {
        "total_labels": len(simple_labels),
        "total_classes": len(all_classes),
        "total_properties": len(all_properties),
        "total_hierarchy_entries": len(all_hierarchy),
        "total_property_metadata": len(all_property_metadata)
    },
    "labels": simple_labels,
    "labels_by_language": all_labels,
    "classes": sorted(list(all_classes)),
    "properties": sorted(list(all_properties)),
    "class_hierarchy": all_hierarchy,
    "property_metadata": all_property_metadata
}

# Save to OneLake
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

file_size_kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f"✓ Saved ontology metadata to: {OUTPUT_PATH}")
print(f"  File size: {file_size_kb:.1f} KB")

In [ ]:
# Verification: Show sample hierarchy
print("Sample class hierarchy:")
for cls, parents in list(all_hierarchy.items())[:5]:
    cls_label = simple_labels.get(cls, cls.split("/")[-1].split("#")[-1])
    parent_labels = [simple_labels.get(p, p.split("/")[-1].split("#")[-1]) for p in parents[:3]]
    print(f"  {cls_label} → {', '.join(parent_labels)}")

print("\nSample property domains/ranges:")
for prop, meta in list(all_property_metadata.items())[:5]:
    prop_label = simple_labels.get(prop, prop.split("/")[-1].split("#")[-1])
    domains = [simple_labels.get(d, d.split("/")[-1].split("#")[-1]) for d in meta["domains"][:2]]
    ranges = [simple_labels.get(r, r.split("/")[-1].split("#")[-1]) for r in meta["ranges"][:2]]
    print(f"  {prop_label}:")
    if domains:
        print(f"    domain: {', '.join(domains)}")
    if ranges:
        print(f"    range: {', '.join(ranges)}")

In [ ]:
# Utility function for other notebooks to load this metadata
def load_ontology_metadata(path: str = OUTPUT_PATH) -> dict:
    """
    Load ontology metadata from JSON file.
    Use this function in other notebooks:
    
    ```python
    %run 13_ontology_enrichment.ipynb
    metadata = load_ontology_metadata()
    label = metadata['labels'].get(uri, uri)
    ```
    """
    if not os.path.exists(path):
        print(f"Ontology metadata not found at {path}")
        print("Run 12_external_ontology_fetcher.ipynb and 13_ontology_enrichment.ipynb first.")
        return {}
    
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def get_label(uri: str, metadata: dict = None) -> str:
    """
    Get human-readable label for a URI.
    Falls back to local name if no label found.
    """
    if metadata is None:
        metadata = output  # Use in-memory if available
    
    labels = metadata.get("labels", {})
    if uri in labels:
        return labels[uri]
    
    # Fall back to local name
    if "#" in uri:
        return uri.split("#")[-1]
    return uri.split("/")[-1]

def get_parent_classes(class_uri: str, metadata: dict = None) -> List[str]:
    """Get parent classes for a given class URI."""
    if metadata is None:
        metadata = output
    return metadata.get("class_hierarchy", {}).get(class_uri, [])

def get_property_domain_range(prop_uri: str, metadata: dict = None) -> Tuple[List[str], List[str]]:
    """Get domains and ranges for a property URI."""
    if metadata is None:
        metadata = output
    prop_meta = metadata.get("property_metadata", {}).get(prop_uri, {})
    return prop_meta.get("domains", []), prop_meta.get("ranges", [])

print("Utility functions defined for use in other notebooks.")
print("\nExample usage:")
print('  metadata = load_ontology_metadata()')
print('  label = get_label("http://qudt.org/schema/qudt/Unit", metadata)')
print('  parents = get_parent_classes("http://qudt.org/schema/qudt/Unit", metadata)')

## Summary

This notebook extracted semantic metadata from cached external ontologies:

- **Labels**: Human-readable names for classes and properties (`rdfs:label`, `skos:prefLabel`)
- **Class hierarchy**: Parent-child relationships (`rdfs:subClassOf`)
- **Property metadata**: Domain and range constraints (`rdfs:domain`, `rdfs:range`)

### Next steps

Use this metadata in:
1. `02_schema_detector.ipynb` - Show labels in detected schema overview
2. `04_property_mapping.ipynb` - Use labels for property names
3. `07_ontology_definition_generator.ipynb` - Include labels in Fabric Ontology

### Output file

`Files/cache/ontology_metadata.json` contains all extracted metadata.